# Rental Inquiry Suitability Classifier
**Objective:** Classify rental inquiries as Suitable (1) or Not Suitable (0) based on landlord preferences, and rank candidates by suitability probability.

**Models:** Naïve Bayes, Decision Tree (TF-IDF), BERT (Contextual Embeddings)

**Dataset:** 210 Daft.ie email enquiries for Apartment 14, Iveagh Court, Dublin 2

## 1. Setup & Install Dependencies

In [ ]:
# Install transformers for BERT (required in Google Colab)
!pip install transformers -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, precision_score,
    recall_score, accuracy_score
)
from scipy.sparse import hstack

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

print('All imports successful.')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Load Data

In [ ]:
# Download datasets from GitHub
import urllib.request

bert_url = 'https://github.com/B00766693/College_Rental_Suitability/raw/main/CompleteDatabase%20for%20BERT.xlsx'
nbdt_url = 'https://github.com/B00766693/College_Rental_Suitability/raw/main/CompleteDatabase%20for%20NB%20and%20DT.xlsx'

urllib.request.urlretrieve(bert_url, 'CompleteDatabase for BERT.xlsx')
urllib.request.urlretrieve(nbdt_url, 'CompleteDatabase for NB and DT.xlsx')

print('Files downloaded successfully.')

In [ ]:
# Load both datasets
df_bert = pd.read_excel('CompleteDatabase for BERT.xlsx')
df_nbdt = pd.read_excel('CompleteDatabase for NB and DT.xlsx')

print('BERT dataset:')
print(f'  Shape: {df_bert.shape}')
print(f'  Columns: {list(df_bert.columns)}')
print()
print('NB/DT dataset:')
print(f'  Shape: {df_nbdt.shape}')
print(f'  Columns: {list(df_nbdt.columns)}')
print()
df_bert.head(5)

In [ ]:
df_nbdt.head(5)

In [ ]:
# Check for missing values
print('Missing values (BERT):')
print(df_bert.isnull().sum())
print()
print('Missing values (NB/DT):')
print(df_nbdt.isnull().sum())
print()
print('Data types (BERT):')
print(df_bert.dtypes)

## 3. Feature Engineering — Derived/Synthetic Features
Derived from the raw message text (BERT file) using regex and keyword matching.
These features address specific landlord constraints and agent preferences.

In [ ]:
def extract_derived_features(df):
    """Extract synthetic features from raw message text."""
    df = df.copy()

    # --- Total_Occupancy_Count ---
    # Start with num_adults; override if message indicates 2 people
    couple_pattern = r'\bmy partner\b|\bmy husband\b|\bmy wife\b|\bmy fianc|\bcouple\b|\btwo of us\b|\bboth of us\b|\bwe are\b.*\blooking\b|\bmyself and my\b'
    df['Total_Occupancy_Count'] = df['num_adults']
    couple_mask = df['message_snippet'].str.contains(couple_pattern, case=False, regex=True, na=False)
    df.loc[couple_mask & (df['num_adults'] == 1), 'Total_Occupancy_Count'] = 2

    # --- Pet_Binary ---
    # Start with pets_info; override if message mentions pets
    pet_pattern = r'\bmy dog\b|\bmy cat\b|\bhave a dog\b|\bhave a cat\b|\bhave pets\b|\bpet deposit\b|\bsmall dog\b|\bsmall cat\b'
    df['Pet_Binary'] = df['pets_info']
    pet_msg_mask = df['message_snippet'].str.contains(pet_pattern, case=False, regex=True, na=False)
    df.loc[pet_msg_mask, 'Pet_Binary'] = 1

    # --- Professional_Status ---
    # Identify professional employment markers
    professional_pattern = (
        r'\baccountant\b|\bengineer\b|\bsolicitor\b|\bdentist\b|\barchitect\b|\blecturer\b|\bconsultant\b|\banalyst\b|'
        r'\bscientist\b|\bdeveloper\b|\bmanager\b|\bdesigner\b|\bpharmacist\b|\bnurse\b|\bdoctor\b|\bpilot\b|'
        r'\bgoogle\b|\bmeta\b|\bmicrosoft\b|\baccenture\b|\bamazon\b|\bsalesforce\b|\bpwc\b|\bdeloitte\b|\bkpmg\b|\bey\b|'
        r'\bmckinsey\b|\bstripe\b|\bhubspot\b|\bintercom\b|\bworkday\b|\bzendesk\b|\bshopify\b|\bmastercard\b|'
        r'\bfull[\-\s]?time\b|\bpermanent\b|\bchartered\b'
    )
    df['Professional_Status'] = df['message_snippet'].str.contains(professional_pattern, case=False, regex=True, na=False).astype(int)

    # --- Salary ---
    # Extract salary from message if mentioned (e.g. €45,000 or 45000 euros)
    def extract_salary(text):
        if pd.isna(text):
            return np.nan
        # Match patterns like €45,000 or €45000 or 45,000 euros or earning 45000
        patterns = [
            r'€([\d,]+)',
            r'([\d,]+)\s*(?:euros?|EUR)',
            r'(?:salary|earning|earn|income|make)[^\d]*([\d,]+)',
        ]
        for p in patterns:
            match = re.search(p, str(text), re.IGNORECASE)
            if match:
                raw = match.group(1).replace(',', '')
                if raw.isdigit() and len(raw) > 0:
                    val = int(raw)
                    if 20000 <= val <= 200000:
                        return val
        return np.nan

    df['Salary'] = df['message_snippet'].apply(extract_salary)

    # --- High_Income ---
    # True if salary > 2x minimum (minimum €30k, so threshold = €60k)
    MINIMUM_SALARY = 30000
    df['High_Income'] = (df['Salary'] >= 2 * MINIMUM_SALARY).astype(int)
    # Neutral value (0) for those without salary info
    df.loc[df['Salary'].isna(), 'High_Income'] = 0

    return df

# Extract derived features from the BERT file (raw text)
df_bert = extract_derived_features(df_bert)

# Copy derived features to the NB/DT dataframe (same row order, same IDs)
for col in ['Total_Occupancy_Count', 'Pet_Binary', 'Professional_Status', 'Salary', 'High_Income']:
    df_nbdt[col] = df_bert[col].values

print('Derived features added to both DataFrames.')
print()
print(df_bert[['num_adults', 'Total_Occupancy_Count', 'pets_info', 'Pet_Binary',
               'Professional_Status', 'Salary', 'High_Income', 'decision']].describe())

In [ ]:
# Show a few examples of the derived features
print('Sample rows with derived features:')
df_bert[['num_adults', 'Total_Occupancy_Count', 'pets_info', 'Pet_Binary',
          'Professional_Status', 'Salary', 'High_Income', 'decision']].sample(10, random_state=42)

In [ ]:
# Check for overrides: where derived features differ from original form fields

# Total_Occupancy_Count vs num_adults
occ_diff = df_bert[df_bert['num_adults'] != df_bert['Total_Occupancy_Count']]
print(f'Occupancy overrides (message indicates more people than num_adults): {len(occ_diff)}')
if len(occ_diff) > 0:
    print(occ_diff[['num_adults', 'Total_Occupancy_Count', 'decision', 'message_snippet']].assign(
        message_snippet=lambda x: x['message_snippet'].str[:120] + '...'
    ).to_string(index=True))

print()

# Pet_Binary vs pets_info
pet_diff = df_bert[df_bert['pets_info'] != df_bert['Pet_Binary']]
print(f'Pet overrides (message mentions pets despite pets_info=0): {len(pet_diff)}')
if len(pet_diff) > 0:
    print(pet_diff[['pets_info', 'Pet_Binary', 'decision', 'message_snippet']].assign(
        message_snippet=lambda x: x['message_snippet'].str[:120] + '...'
    ).to_string(index=True))

## 4. Exploratory Data Analysis & Insights

In [ ]:
# Class distribution
colors = ['#22c55e', '#ef4444']

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_bert, x='decision', palette=colors, hue='decision', legend=False)
plt.title('Distribution of Suitability Decision')
plt.ylabel('Number of Inquiries')
plt.xticks([0, 1], ['Not Suitable (0)', 'Suitable (1)'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.savefig('class_distribution.pdf', format='pdf', bbox_inches='tight')
plt.show()

print(f'Suitable: {(df_bert["decision"]==1).sum()} ({(df_bert["decision"]==1).mean()*100:.1f}%)')
print(f'Not Suitable: {(df_bert["decision"]==0).sum()} ({(df_bert["decision"]==0).mean()*100:.1f}%)')

In [ ]:
# Adults per response with decision breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Adults by decision
ct = pd.crosstab(df_bert['Total_Occupancy_Count'], df_bert['decision'])
ct.columns = ['Not Suitable', 'Suitable']
ct.plot(kind='bar', ax=axes[0], color=['#ef4444', '#22c55e'], edgecolor='white')
axes[0].set_title('Decision by Occupancy Count')
axes[0].set_xlabel('Total Occupancy Count')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend()

# Chart 2: Pets by decision
ct2 = pd.crosstab(df_bert['Pet_Binary'], df_bert['decision'])
ct2.index = ['No Pets', 'Has Pets']
ct2.columns = ['Not Suitable', 'Suitable']
ct2.plot(kind='bar', ax=axes[1], color=['#ef4444', '#22c55e'], edgecolor='white')
axes[1].set_title('Decision by Pet Status')
axes[1].set_xlabel('Pet Status')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend()

plt.tight_layout()
plt.savefig('decision_by_features.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Professional status and salary insights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Professional status by decision
ct3 = pd.crosstab(df_bert['Professional_Status'], df_bert['decision'])
ct3.index = ['Not Professional', 'Professional']
ct3.columns = ['Not Suitable', 'Suitable']
ct3.plot(kind='bar', ax=axes[0], color=['#ef4444', '#22c55e'], edgecolor='white')
axes[0].set_title('Decision by Professional Status')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].set_ylabel('Count')

# Salary distribution for those who stated it
salary_data = df_bert[df_bert['Salary'].notna()]
axes[1].hist(salary_data[salary_data['decision']==1]['Salary'], bins=15, alpha=0.7, label='Suitable', color='#22c55e')
axes[1].hist(salary_data[salary_data['decision']==0]['Salary'], bins=15, alpha=0.7, label='Not Suitable', color='#ef4444')
axes[1].set_title('Salary Distribution by Decision')
axes[1].set_xlabel('Salary (€)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('professional_salary_insights.pdf', format='pdf', bbox_inches='tight')
plt.show()

print(f'Applicants with salary stated: {salary_data.shape[0]} ({salary_data.shape[0]/210*100:.1f}%)')
print(f'High income applicants: {(df_bert["High_Income"]==1).sum()}')

## 5. Prepare Data for Traditional Classifiers (Naïve Bayes & Decision Tree)
Feature extraction: TF-IDF on preprocessed message text + structured/derived features

In [ ]:
# Drop non-predictive columns
drop_cols = ['id', 'property_id']
df_ml = df_nbdt.drop(columns=drop_cols)

# Target variable
y = df_ml['decision']

# TF-IDF on the preprocessed message text
tfidf = TfidfVectorizer(max_features=500)
X_tfidf = tfidf.fit_transform(df_ml['message_snippet'].fillna(''))

print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'Top 20 TF-IDF features: {tfidf.get_feature_names_out()[:20]}')

In [ ]:
# Combine TF-IDF with structured and derived features
structured_features = ['num_adults', 'pets_info', 'Total_Occupancy_Count',
                        'Pet_Binary', 'Professional_Status', 'High_Income']

X_structured = df_ml[structured_features].fillna(0).values
X_structured_sparse = scipy_sparse = hstack([X_tfidf, X_structured])

print(f'Combined feature matrix shape: {X_structured_sparse.shape}')
print(f'  TF-IDF features: {X_tfidf.shape[1]}')
print(f'  Structured features: {len(structured_features)}')

In [ ]:
# 70/30 train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_structured_sparse, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Train class distribution: {dict(y_train.value_counts())}')
print(f'Test class distribution: {dict(y_test.value_counts())}')

## 6. Naïve Bayes Classifier

In [ ]:
# Train Naïve Bayes with 5-fold cross-validation
nb_model = MultinomialNB()

# 5-fold cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(nb_model, X_train, y_train, cv=cv, scoring='f1')

print('Naïve Bayes — 5-Fold Cross-Validation (F1 Score):')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

In [ ]:
# Fit on full training set and evaluate on test set
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_proba = nb_model.predict_proba(X_test)[:, 1]

print('Naïve Bayes — Test Set Results:')
print(classification_report(y_test, nb_pred, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(y_test, nb_proba):.4f}')

In [ ]:
# Confusion Matrix — Naïve Bayes
cm_nb = confusion_matrix(y_test, nb_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Suitable', 'Suitable'],
            yticklabels=['Not Suitable', 'Suitable'])
plt.title('Confusion Matrix — Naïve Bayes')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix_nb.pdf', format='pdf', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_nb.ravel()
print(f'True Positives: {tp}, True Negatives: {tn}')
print(f'False Positives: {fp} (unsuitable invited), False Negatives: {fn} (suitable rejected)')

### Naïve Bayes — Observations

**Key findings from the Naïve Bayes classifier:**

- The confusion matrix reveals the balance between false positives (unsuitable candidates incorrectly invited) and false negatives (suitable candidates missed). In the context of this rental screening system, **false positives are more costly** — they waste the agent's time with unsuitable viewings.
- Naïve Bayes assumes feature independence, which means it treats each TF-IDF term independently. This is a reasonable assumption for keyword-based signals (e.g., 'pet', 'partner', 'couple') but may miss contextual phrases where word order matters (e.g., 'I have **no** pets' vs 'I have pets').
- The 5-fold cross-validation scores indicate how stable the model's performance is across different subsets of the training data. High variance between folds would suggest overfitting to particular samples.
- As a probabilistic classifier, NB naturally produces well-calibrated probability scores, which is valuable for the candidate ranking objective.

In [ ]:
# Top features for Naïve Bayes — most indicative words per class
feature_names_nb = list(tfidf.get_feature_names_out()) + structured_features
log_probs = nb_model.feature_log_prob_

# Top words for 'Suitable' (class 1) vs 'Not Suitable' (class 0)
log_ratio = log_probs[1] - log_probs[0]  # positive = favours Suitable

top_suitable_idx = np.argsort(log_ratio)[-15:]
top_not_suitable_idx = np.argsort(log_ratio)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh([feature_names_nb[i] for i in top_suitable_idx],
             log_ratio[top_suitable_idx], color='#22c55e')
axes[0].set_title('Top 15 Features → Suitable')
axes[0].set_xlabel('Log Probability Ratio')

axes[1].barh([feature_names_nb[i] for i in top_not_suitable_idx],
             log_ratio[top_not_suitable_idx], color='#ef4444')
axes[1].set_title('Top 15 Features → Not Suitable')
axes[1].set_xlabel('Log Probability Ratio')

plt.suptitle('Naïve Bayes — Most Indicative Features per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_nb.pdf', format='pdf', bbox_inches='tight')
plt.show()

## 7. Decision Tree Classifier

In [ ]:
# Train Decision Tree with 5-fold cross-validation
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_leaf=5)

cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=cv, scoring='f1')

print('Decision Tree — 5-Fold Cross-Validation (F1 Score):')
for i, score in enumerate(cv_scores_dt, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean F1: {cv_scores_dt.mean():.4f} (+/- {cv_scores_dt.std():.4f})')

In [ ]:
# Fit on full training set and evaluate on test set
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

print('Decision Tree — Test Set Results:')
print(classification_report(y_test, dt_pred, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(y_test, dt_proba):.4f}')

In [ ]:
# Confusion Matrix — Decision Tree
cm_dt = confusion_matrix(y_test, dt_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not Suitable', 'Suitable'],
            yticklabels=['Not Suitable', 'Suitable'])
plt.title('Confusion Matrix — Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix_dt.pdf', format='pdf', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_dt.ravel()
print(f'True Positives: {tp}, True Negatives: {tn}')
print(f'False Positives: {fp} (unsuitable invited), False Negatives: {fn} (suitable rejected)')

In [ ]:
# Visualise the Decision Tree (top levels for explainability)
plt.figure(figsize=(20, 10))
feature_names = list(tfidf.get_feature_names_out()) + structured_features
plot_tree(dt_model, feature_names=feature_names, class_names=['Not Suitable', 'Suitable'],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title('Decision Tree — Top 3 Levels')
plt.savefig('decision_tree_visualisation.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 most important features in the Decision Tree
importances = dt_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df, x='importance', y='feature', palette='viridis')
plt.title('Top 10 Feature Importances — Decision Tree')
plt.xlabel('Importance')
plt.savefig('feature_importance_dt.pdf', format='pdf', bbox_inches='tight')
plt.show()

### Decision Tree — Observations

**Key findings from the Decision Tree classifier:**

- The tree visualisation above reveals the **exact rules** driving suitability decisions. This is the primary advantage of the Decision Tree — full explainability. An agent can see precisely why a candidate was classified as suitable or unsuitable.
- The feature importance chart shows which attributes have the most discriminative power. Features like `Total_Occupancy_Count` and `Pet_Binary` are expected to rank highly given the landlord's strict single-occupancy and no-pets rules.
- If TF-IDF text features dominate the importance rankings, it suggests the unstructured message text carries significant signal beyond the structured form fields — validating the dual-source feature approach.
- The Decision Tree may be prone to overfitting on a small dataset (n=210). The `max_depth=10` and `min_samples_leaf=5` constraints help mitigate this, and the cross-validation scores provide evidence of generalisation ability.

## 8. ROC Curves & Precision-Recall — NB vs DT Comparison

In [ ]:
# ROC Curves — NB vs DT
fpr_nb, tpr_nb, _ = roc_curve(y_test, nb_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba)
auc_nb = roc_auc_score(y_test, nb_proba)
auc_dt = roc_auc_score(y_test, dt_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_nb, tpr_nb, color='blue', label=f'Naïve Bayes (AUC = {auc_nb:.2f})')
plt.plot(fpr_dt, tpr_dt, color='green', label=f'Decision Tree (AUC = {auc_dt:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — Naïve Bayes vs Decision Tree')
plt.legend()
plt.grid(True)
plt.savefig('roc_curve_nb_dt.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall Curves — NB vs DT
precision_nb, recall_nb, thresholds_nb = precision_recall_curve(y_test, nb_proba)
precision_dt, recall_dt, thresholds_dt = precision_recall_curve(y_test, dt_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall_nb, precision_nb, color='blue', label='Naïve Bayes')
plt.plot(recall_dt, precision_dt, color='green', label='Decision Tree')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Naïve Bayes vs Decision Tree')
plt.legend()
plt.grid(True)
plt.savefig('precision_recall_nb_dt.pdf', format='pdf', bbox_inches='tight')
plt.show()

## 9. BERT Classifier
Using raw text with contextual word embeddings. BERT preserves full linguistic context including casing, punctuation, and stop words.

In [ ]:
# Prepare BERT data
df_bert_ml = df_bert.drop(columns=['id', 'property_id'])

texts = df_bert_ml['message_snippet'].fillna('').tolist()
labels = df_bert_ml['decision'].tolist()
structured_vals = df_bert_ml[structured_features].fillna(0).values

# 70/30 split (same random state for comparability)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)
train_struct, test_struct, _, _ = train_test_split(
    structured_vals, labels, test_size=0.3, random_state=42, stratify=labels
)

print(f'BERT Training samples: {len(train_texts)}')
print(f'BERT Test samples: {len(test_texts)}')

In [ ]:
# BERT Tokenisation (WordPiece)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN = 256

class RentalDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = RentalDataset(train_texts, train_labels)
test_dataset = RentalDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

# Show tokenisation example
sample = tokenizer(train_texts[0], max_length=MAX_LEN, truncation=True)
print(f'\nSample tokenisation (first 20 tokens):')
print(tokenizer.convert_ids_to_tokens(sample['input_ids'][:20]))

In [ ]:
# Load pre-trained BERT and set up for fine-tuning
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

# Optimizer and scheduler
EPOCHS = 4
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print(f'Total training steps: {total_steps}')
print(f'Epochs: {EPOCHS}, Learning rate: {LEARNING_RATE}')

In [ ]:
# Fine-tune BERT
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} — Average Loss: {avg_loss:.4f}')

print('\nBERT fine-tuning complete.')

In [ ]:
# Training loss curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS+1), train_losses, marker='o', color='#6366f1', linewidth=2)
plt.title('BERT Training Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig('bert_training_loss.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate BERT on test set
model.eval()
bert_preds = []
bert_proba = []
bert_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)

        bert_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        bert_proba.extend(probs[:, 1].cpu().numpy())
        bert_true.extend(labels.numpy())

print('BERT — Test Set Results:')
print(classification_report(bert_true, bert_preds, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(bert_true, bert_proba):.4f}')

In [ ]:
# Confusion Matrix — BERT
cm_bert = confusion_matrix(bert_true, bert_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Not Suitable', 'Suitable'],
            yticklabels=['Not Suitable', 'Suitable'])
plt.title('Confusion Matrix — BERT')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix_bert.pdf', format='pdf', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_bert.ravel()
print(f'True Positives: {tp}, True Negatives: {tn}')
print(f'False Positives: {fp} (unsuitable invited), False Negatives: {fn} (suitable rejected)')

In [ ]:
# Precision-Recall Curve — BERT
precision_bert, recall_bert, thresholds_bert = precision_recall_curve(bert_true, bert_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall_nb, precision_nb, color='blue', label='Naïve Bayes')
plt.plot(recall_dt, precision_dt, color='green', label='Decision Tree')
plt.plot(recall_bert, precision_bert, color='purple', label='BERT')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — All Three Models')
plt.legend()
plt.grid(True)
plt.savefig('precision_recall_all_models.pdf', format='pdf', bbox_inches='tight')
plt.show()

### BERT — Observations

**Key findings from the BERT classifier:**

- BERT's contextual embeddings allow it to understand the **intent** behind phrases — critically, distinguishing 'I have no pets' from 'I have pets', which traditional bag-of-words models may struggle with if negations are not perfectly preserved.
- The training loss curve should show a decreasing trend across epochs. If loss plateaus early, the model may have learned the key patterns quickly given the relatively small dataset size (n=210).
- With only 210 samples, BERT is at risk of overfitting to the training data. The test set performance is the key indicator of whether the model has generalised.
- **Note on cross-validation:** Unlike NB and DT, 5-fold cross-validation was not performed for BERT due to the significant computational cost of fine-tuning a transformer model five times. The 70/30 holdout evaluation provides the primary performance assessment.
- BERT's probability scores enable candidate ranking, similar to the traditional classifiers, but the scores are derived from contextual understanding of the full message rather than isolated keyword frequencies.

### Threshold Optimisation
The default classification threshold is 0.5. However, since **false positives (inviting unsuitable candidates) are more costly** than false negatives, the threshold can be adjusted upward to prioritise precision. The Precision-Recall curve is used to identify the optimal operating point.

In [ ]:
# Threshold optimisation — prioritise precision (minimise false positives)
from sklearn.metrics import precision_recall_curve, f1_score

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_thresh = [
    ('Naïve Bayes', y_test, nb_proba, 'blue'),
    ('Decision Tree', y_test, dt_proba, 'green'),
    ('BERT', bert_true, bert_proba, 'purple')
]

optimal_thresholds = {}

for ax, (name, true, proba, color) in zip(axes, models_thresh):
    precisions, recalls, thresholds = precision_recall_curve(true, proba)
    
    # Calculate F1 at each threshold
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    
    # Find threshold that maximises F1 while keeping precision >= 0.8
    high_prec_mask = precisions[:-1] >= 0.8
    if high_prec_mask.any():
        best_idx = np.argmax(f1_scores * high_prec_mask)
    else:
        best_idx = np.argmax(f1_scores)
    
    opt_thresh = thresholds[best_idx]
    optimal_thresholds[name] = opt_thresh
    
    ax.plot(thresholds, precisions[:-1], label='Precision', color='blue')
    ax.plot(thresholds, recalls[:-1], label='Recall', color='red')
    ax.plot(thresholds, f1_scores, label='F1', color='green', linestyle='--')
    ax.axvline(x=opt_thresh, color='gray', linestyle=':', label=f'Optimal: {opt_thresh:.2f}')
    ax.set_title(f'{name}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision / Recall / F1 vs Classification Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_optimisation.pdf', format='pdf', bbox_inches='tight')
plt.show()

print('Optimal thresholds (maximising F1 with precision >= 0.8):')
for name, thresh in optimal_thresholds.items():
    print(f'  {name}: {thresh:.4f}')

In [ ]:
# Apply optimised thresholds and compare with default (0.5)
print('Performance at DEFAULT threshold (0.5) vs OPTIMISED threshold:')
print('=' * 75)

for name, true, proba in [('Naïve Bayes', y_test.values, nb_proba),
                           ('Decision Tree', y_test.values, dt_proba),
                           ('BERT', np.array(bert_true), np.array(bert_proba))]:
    
    opt_thresh = optimal_thresholds[name]
    pred_default = (np.array(proba) >= 0.5).astype(int)
    pred_optimised = (np.array(proba) >= opt_thresh).astype(int)
    
    print(f'\n{name} (optimal threshold: {opt_thresh:.4f}):')
    print(f'  {"Metric":<12} {"Default (0.5)":>14} {"Optimised":>14}')
    print(f'  {"Precision":<12} {precision_score(true, pred_default):>14.4f} {precision_score(true, pred_optimised):>14.4f}')
    print(f'  {"Recall":<12} {recall_score(true, pred_default):>14.4f} {recall_score(true, pred_optimised):>14.4f}')
    print(f'  {"F1":<12} {f1_score(true, pred_default):>14.4f} {f1_score(true, pred_optimised):>14.4f}')
    
    cm_opt = confusion_matrix(true, pred_optimised)
    tn, fp, fn, tp = cm_opt.ravel()
    print(f'  False Positives (costly): {fp}')

### Error Analysis — Examining Misclassified Cases
Understanding **why** models make errors is critical. False positives (unsuitable candidates invited) are the most disruptive for the letting agent. This analysis examines the specific cases each model misclassified.

In [ ]:
# Error analysis — examine misclassified cases
print('=' * 80)
print('ERROR ANALYSIS — Naïve Bayes')
print('=' * 80)

# False positives: predicted suitable but actually not suitable
fp_mask_nb = (nb_pred == 1) & (y_test.values == 0)
fp_indices_nb = y_test.index[fp_mask_nb]
print(f'\nFalse Positives (unsuitable invited): {fp_mask_nb.sum()}')
if fp_mask_nb.sum() > 0:
    for idx in fp_indices_nb:
        row = df_bert.loc[idx]
        print(f'  Row {idx}: num_adults={row["num_adults"]}, pets={row["pets_info"]}, '
              f'occupancy={row["Total_Occupancy_Count"]}, pet_binary={row["Pet_Binary"]}')
        print(f'    Message: {str(row["message_snippet"])[:150]}...')

# False negatives: predicted not suitable but actually suitable
fn_mask_nb = (nb_pred == 0) & (y_test.values == 1)
fn_indices_nb = y_test.index[fn_mask_nb]
print(f'\nFalse Negatives (suitable rejected): {fn_mask_nb.sum()}')
if fn_mask_nb.sum() > 0:
    for idx in fn_indices_nb[:5]:  # show up to 5
        row = df_bert.loc[idx]
        print(f'  Row {idx}: num_adults={row["num_adults"]}, pets={row["pets_info"]}, '
              f'professional={row["Professional_Status"]}')
        print(f'    Message: {str(row["message_snippet"])[:150]}...')

In [ ]:
# Error analysis — Decision Tree and BERT
print('=' * 80)
print('ERROR ANALYSIS — Decision Tree')
print('=' * 80)

fp_mask_dt = (dt_pred == 1) & (y_test.values == 0)
fn_mask_dt = (dt_pred == 0) & (y_test.values == 1)
print(f'False Positives: {fp_mask_dt.sum()}')
print(f'False Negatives: {fn_mask_dt.sum()}')

if fp_mask_dt.sum() > 0:
    print('\nFalse Positive details:')
    for idx in y_test.index[fp_mask_dt]:
        row = df_bert.loc[idx]
        print(f'  Row {idx}: occupancy={row["Total_Occupancy_Count"]}, pet_binary={row["Pet_Binary"]}, '
              f'professional={row["Professional_Status"]}')
        print(f'    Message: {str(row["message_snippet"])[:150]}...')

print()
print('=' * 80)
print('ERROR ANALYSIS — BERT')
print('=' * 80)

bert_preds_arr = np.array(bert_preds)
bert_true_arr = np.array(bert_true)
fp_mask_bert = (bert_preds_arr == 1) & (bert_true_arr == 0)
fn_mask_bert = (bert_preds_arr == 0) & (bert_true_arr == 1)
print(f'False Positives: {fp_mask_bert.sum()}')
print(f'False Negatives: {fn_mask_bert.sum()}')

# Compare error overlap between models
print()
print('=' * 80)
print('ERROR OVERLAP ANALYSIS')
print('=' * 80)
print(f'Cases misclassified by ALL 3 models: '
      f'{((nb_pred != y_test.values) & (dt_pred != y_test.values) & (bert_preds_arr != bert_true_arr)).sum()}')
print(f'Cases misclassified by NB only: '
      f'{((nb_pred != y_test.values) & (dt_pred == y_test.values)).sum()}')
print(f'Cases misclassified by DT only: '
      f'{((dt_pred != y_test.values) & (nb_pred == y_test.values)).sum()}')

**Error Analysis Observations:**

- False positives typically arise from edge cases where the structured fields suggest suitability (num_adults=1, no pets) but the message text contains disqualifying information (e.g., mentions of a partner, student status, or unemployment).
- If certain cases are consistently misclassified across all three models, they likely represent genuinely ambiguous applications where the boundary between suitable and unsuitable is unclear.
- Cases misclassified by only one model reveal that model's specific weakness — e.g., NB may miss contextual negations while DT may miss nuanced text signals not captured in its top features.

## 10. Model Comparison — All Three Models

In [ ]:
# ROC Curves — All three models
fpr_bert, tpr_bert, _ = roc_curve(bert_true, bert_proba)
auc_bert = roc_auc_score(bert_true, bert_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_nb, tpr_nb, color='blue', label=f'Naïve Bayes (AUC = {auc_nb:.2f})')
plt.plot(fpr_dt, tpr_dt, color='green', label=f'Decision Tree (AUC = {auc_dt:.2f})')
plt.plot(fpr_bert, tpr_bert, color='purple', label=f'BERT (AUC = {auc_bert:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — Model Comparison')
plt.legend()
plt.grid(True)
plt.savefig('roc_curve_all_models.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Model': ['Naïve Bayes', 'Decision Tree', 'BERT'],
    'Accuracy': [
        accuracy_score(y_test, nb_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(bert_true, bert_preds)
    ],
    'Precision': [
        precision_score(y_test, nb_pred),
        precision_score(y_test, dt_pred),
        precision_score(bert_true, bert_preds)
    ],
    'Recall': [
        recall_score(y_test, nb_pred),
        recall_score(y_test, dt_pred),
        recall_score(bert_true, bert_preds)
    ],
    'F1 Score': [
        f1_score(y_test, nb_pred),
        f1_score(y_test, dt_pred),
        f1_score(bert_true, bert_preds)
    ],
    'ROC AUC': [auc_nb, auc_dt, auc_bert]
})

comparison = comparison.round(4)
print('Model Comparison Summary:')
print('=' * 70)
print(comparison.to_string(index=False))

In [ ]:
# Confusion Matrix Summary — all three models side by side
cm_data = []
for name, true, preds in [('Naïve Bayes', y_test.values, nb_pred),
                           ('Decision Tree', y_test.values, dt_pred),
                           ('BERT', np.array(bert_true), np.array(bert_preds))]:
    cm = confusion_matrix(true, preds)
    tn, fp, fn, tp = cm.ravel()
    cm_data.append({'Model': name, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
                    'FP Rate': f'{fp/(fp+tn)*100:.1f}%', 'FN Rate': f'{fn/(fn+tp)*100:.1f}%'})

cm_summary_df = pd.DataFrame(cm_data)
print('Confusion Matrix Summary — All Models')
print('=' * 75)
print(cm_summary_df.to_string(index=False))
print()
print('FP = unsuitable invited (costly to agent)')
print('FN = suitable missed (lost opportunity)')

# Cost-weighted analysis: FP costs 3x more than FN
FP_COST = 3
FN_COST = 1
print(f'\nCost-Weighted Error Analysis (FP cost={FP_COST}x, FN cost={FN_COST}x):')
print('-' * 50)
for _, row in cm_summary_df.iterrows():
    total_cost = row['FP'] * FP_COST + row['FN'] * FN_COST
    print(f'  {row["Model"]}: FP({row["FP"]})×{FP_COST} + FN({row["FN"]})×{FN_COST} = Total Cost: {total_cost}')

In [ ]:
# Prediction confidence distribution — how decisive are the models?
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, proba, true, color) in zip(axes, [
    ('Naïve Bayes', nb_proba, y_test.values, 'blue'),
    ('Decision Tree', dt_proba, y_test.values, 'green'),
    ('BERT', np.array(bert_proba), np.array(bert_true), 'purple')
]):
    # Separate by actual class
    proba_suitable = np.array(proba)[np.array(true) == 1]
    proba_not_suitable = np.array(proba)[np.array(true) == 0]

    ax.hist(proba_suitable, bins=15, alpha=0.7, label='Actually Suitable', color='#22c55e', edgecolor='white')
    ax.hist(proba_not_suitable, bins=15, alpha=0.7, label='Actually Not Suitable', color='#ef4444', edgecolor='white')
    ax.axvline(x=0.5, color='gray', linestyle='--', label='Threshold (0.5)')
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel('Predicted Probability (Suitable)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Prediction Confidence Distribution — How Decisive Are the Models?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_distribution.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Summary stats
for name, proba in [('Naïve Bayes', nb_proba), ('Decision Tree', dt_proba), ('BERT', bert_proba)]:
    p = np.array(proba)
    high_conf = ((p > 0.9) | (p < 0.1)).sum()
    uncertain = ((p > 0.4) & (p < 0.6)).sum()
    print(f'{name}: {high_conf} high-confidence ({high_conf/len(p)*100:.0f}%), '
          f'{uncertain} uncertain ({uncertain/len(p)*100:.0f}%)')

## 11. Candidate Ranking — Top 10 Shortlist
Rank test set candidates by suitability probability score to identify the best shortlist.

In [ ]:
# Build ranking dataframe from test set
test_indices = y_test.index.tolist()

ranking_df = pd.DataFrame({
    'Index': test_indices,
    'Actual': y_test.values,
    'NB_Probability': nb_proba,
    'DT_Probability': dt_proba,
})

# Add BERT probabilities (aligned to same test split via random_state=42)
ranking_df['BERT_Probability'] = bert_proba

# Average ensemble probability
ranking_df['Ensemble_Probability'] = (
    ranking_df['NB_Probability'] + ranking_df['DT_Probability'] + ranking_df['BERT_Probability']
) / 3

In [ ]:
# Top 10 candidates by each model
print('=' * 80)
print('TOP 10 CANDIDATES — Naïve Bayes (by suitability probability):')
print('=' * 80)
top10_nb = ranking_df.nlargest(10, 'NB_Probability')[['Index', 'Actual', 'NB_Probability']]
top10_nb['Rank'] = range(1, 11)
print(top10_nb.to_string(index=False))
print(f'\nCorrectly ranked suitable in Top 10: {(top10_nb["Actual"]==1).sum()}/10')

print()
print('=' * 80)
print('TOP 10 CANDIDATES — Decision Tree (by suitability probability):')
print('=' * 80)
top10_dt = ranking_df.nlargest(10, 'DT_Probability')[['Index', 'Actual', 'DT_Probability']]
top10_dt['Rank'] = range(1, 11)
print(top10_dt.to_string(index=False))
print(f'\nCorrectly ranked suitable in Top 10: {(top10_dt["Actual"]==1).sum()}/10')

print()
print('=' * 80)
print('TOP 10 CANDIDATES — BERT (by suitability probability):')
print('=' * 80)
top10_bert = ranking_df.nlargest(10, 'BERT_Probability')[['Index', 'Actual', 'BERT_Probability']]
top10_bert['Rank'] = range(1, 11)
print(top10_bert.to_string(index=False))
print(f'\nCorrectly ranked suitable in Top 10: {(top10_bert["Actual"]==1).sum()}/10')

In [ ]:
# Ensemble Top 10
print('=' * 80)
print('TOP 10 CANDIDATES — Ensemble Average (all 3 models):')
print('=' * 80)
top10_ens = ranking_df.nlargest(10, 'Ensemble_Probability')
top10_ens['Rank'] = range(1, 11)
print(top10_ens[['Rank', 'Index', 'Actual', 'NB_Probability', 'DT_Probability',
                  'BERT_Probability', 'Ensemble_Probability']].to_string(index=False))
print(f'\nCorrectly ranked suitable in Top 10: {(top10_ens["Actual"]==1).sum()}/10')

In [ ]:
# Visualise Top 10 ranking quality
models = ['Naïve Bayes', 'Decision Tree', 'BERT', 'Ensemble']
correct_in_top10 = [
    (top10_nb['Actual']==1).sum(),
    (top10_dt['Actual']==1).sum(),
    (top10_bert['Actual']==1).sum(),
    (top10_ens['Actual']==1).sum()
]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, correct_in_top10, color=['blue', 'green', 'purple', '#fbbf24'], edgecolor='white')
plt.title('Correctly Suitable Candidates in Top 10 Shortlist')
plt.ylabel('Count (out of 10)')
plt.ylim(0, 11)
for bar, val in zip(bars, correct_in_top10):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2, str(val),
             ha='center', fontweight='bold', fontsize=12)
plt.axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='Perfect (10/10)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.savefig('top10_ranking_quality.pdf', format='pdf', bbox_inches='tight')
plt.show()

## 12. Summary

| Aspect | Naïve Bayes | Decision Tree | BERT |
|--------|------------|---------------|------|
| **Approach** | TF-IDF + structured features | TF-IDF + structured features | Contextual embeddings |
| **Strengths** | Fast, good with keyword signals, well-calibrated probabilities | Fully explainable, reveals exact decision rules | Understands context and intent, handles negation |
| **Weaknesses** | Assumes feature independence, may miss context | Risk of overfitting on small dataset | Computationally expensive, needs GPU |
| **Best for** | Quick baseline, probability ranking | Explainability and auditing | Nuanced text understanding |

**Key Takeaways:**
- The structured features (`Total_Occupancy_Count`, `Pet_Binary`) provide strong baseline separation, as the landlord's rules are largely binary.
- The value of NLP (TF-IDF and BERT) emerges in the edge cases — single applicants whose message text reveals disqualifying information not captured in the form fields.
- Threshold optimisation demonstrates that precision can be increased at a manageable cost to recall, which aligns with the business requirement of minimising unsuitable invitations.
- The Top 10 ranking shows each model's ability to prioritise the most suitable candidates, which is the ultimate business deliverable for the letting agent.

---
## 13. Evaluation Against Different Property Type

**Objective:** Test whether the models and pipeline generalise across different landlord preferences.

- **Property 1** (current): 1-bed apartment, single occupancy, no pets, min salary €30k
- **Property 2** (new): 2-bed apartment, accommodates 2 adults + child (max 3), pets allowed, min salary €40k

This section aims to demonstrate three things:
1. Preferences can be parameterised into a configurable rule engine
2. Models trained on one property's rules do not transfer to different rules
3. The pipeline is reusable, when retrained on new preferences, it produces a working model

### 13a — Configurable Preference Engine
A reusable labelling function that takes landlord preferences as parameters. This is the foundation for a system that can adapt to any property type.

In [ ]:
def label_by_preferences(df, max_occupants=1, pets_allowed=False, min_salary=30000, label='Property'):
    """
    Label applicants as suitable (1) or not suitable (0) based on configurable landlord preferences.
    Uses derived features: Total_Occupancy_Count, Pet_Binary, Professional_Status, Salary.
    """
    df = df.copy()
    decisions = []

    for idx, row in df.iterrows():
        occupancy = row.get('Total_Occupancy_Count', row['num_adults'])
        has_pets = row.get('Pet_Binary', row['pets_info'])
        salary = row.get('Salary', np.nan)
        msg = str(row.get('message_snippet', ''))

        # Rule 1: Occupancy exceeds max
        if occupancy > max_occupants:
            decisions.append(0)
            continue

        # Rule 2: Pets not allowed but applicant has pets
        if not pets_allowed and has_pets == 1:
            decisions.append(0)
            continue

        # Rule 3: Unemployed or student
        if re.search(r'not employed|unemployed|student', msg, re.IGNORECASE):
            decisions.append(0)
            continue

        # Rule 4: Salary below minimum (if stated)
        if pd.notna(salary) and salary < min_salary:
            decisions.append(0)
            continue

        decisions.append(1)

    df[f'decision_{label}'] = decisions
    return df

print('Preference engine function defined.')

In [ ]:
# Apply both property configurations to the existing 210 rows
df_labelled = label_by_preferences(df_bert, max_occupants=1, pets_allowed=False,
                                    min_salary=30000, label='P1')
df_labelled = label_by_preferences(df_labelled, max_occupants=3, pets_allowed=True,
                                    min_salary=40000, label='P2')

# Compare class distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title, colors in [
    (axes[0], 'decision_P1', 'Property 1\n(1-bed, no pets, 1 adult)', ['#ef4444', '#22c55e']),
    (axes[1], 'decision_P2', 'Property 2\n(2-bed, pets OK, max 3)', ['#ef4444', '#22c55e'])
]:
    counts = df_labelled[col].value_counts().sort_index()
    ax.bar(['Not Suitable (0)', 'Suitable (1)'], counts.values, color=colors)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, f'{v} ({v/210*100:.1f}%)', ha='center', fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Count')
    ax.set_ylim(0, 220)

plt.suptitle('Class Distribution Under Different Landlord Preferences (Same 210 Applicants)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('preference_comparison.pdf', format='pdf', bbox_inches='tight')
plt.show()

print(f'Property 1: {(df_labelled["decision_P1"]==1).sum()} suitable / {(df_labelled["decision_P1"]==0).sum()} not suitable')
print(f'Property 2: {(df_labelled["decision_P2"]==1).sum()} suitable / {(df_labelled["decision_P2"]==0).sum()} not suitable')
print(f'\nProperty 2 imbalance ratio: {max((df_labelled["decision_P2"]==1).sum(), (df_labelled["decision_P2"]==0).sum()) / max(1, min((df_labelled["decision_P2"]==1).sum(), (df_labelled["decision_P2"]==0).sum())):.1f}:1')
print('\nThis severe imbalance under Property 2 rules demonstrates that the existing dataset,'
      ' collected for a 1-bed property, is unsuitable for training a Property 2 model.'
      ' A new dataset is needed (Section 13c).')

### 13b — Zero-Shot Transfer Test
Evaluate the models trained on Property 1 preferences against Property 2 labels. This tests whether a model trained on one set of rules can generalise to different rules.

**Expectation:** Poor performance — the Property 1 models learned to reject couples and pet owners, but under Property 2 rules these applicants are suitable.

In [ ]:
# Re-label the test set with Property 2 preferences
test_df = df_labelled.iloc[y_test.index]
y_test_p2 = test_df['decision_P2'].values

# Evaluate Property 1 trained models against Property 2 labels
print('=' * 75)
print('ZERO-SHOT TRANSFER: Property 1 Models → Property 2 Labels')
print('=' * 75)

for name, preds, proba in [('Naïve Bayes', nb_pred, nb_proba),
                            ('Decision Tree', dt_pred, dt_proba)]:
    print(f'\n--- {name} ---')
    print(classification_report(y_test_p2, preds, target_names=['Not Suitable', 'Suitable'],
                                zero_division=0))
    cm = confusion_matrix(y_test_p2, preds)
    tn, fp, fn, tp = cm.ravel()
    print(f'  False Positives: {fp} | False Negatives: {fn}')
    print(f'  Key issue: {fn} suitable applicants (under P2 rules) were rejected by a model'
          f' trained on P1 rules')

# BERT transfer
print(f'\n--- BERT ---')
# BERT test labels need to be aligned with the same test split
bert_true_p2 = y_test_p2  # same indices, different labels
print(classification_report(bert_true_p2, bert_preds, target_names=['Not Suitable', 'Suitable'],
                            zero_division=0))
cm_bert_p2 = confusion_matrix(bert_true_p2, bert_preds)
tn, fp, fn, tp = cm_bert_p2.ravel()
print(f'  False Positives: {fp} | False Negatives: {fn}')
print(f'  Key issue: {fn} suitable applicants (under P2 rules) were rejected')

In [ ]:
# Visualise zero-shot transfer — confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, preds) in zip(axes, [('Naïve Bayes', nb_pred),
                                     ('Decision Tree', dt_pred),
                                     ('BERT', bert_preds)]):
    cm = confusion_matrix(y_test_p2, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
                xticklabels=['Not Suitable', 'Suitable'],
                yticklabels=['Not Suitable', 'Suitable'])
    ax.set_title(f'{name}\n(P1 Model → P2 Labels)', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual (P2 Rules)')

plt.suptitle('Zero-Shot Transfer: Property 1 Models Evaluated with Property 2 Labels',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('zero_shot_transfer_cm.pdf', format='pdf', bbox_inches='tight')
plt.show()

**Zero-Shot Transfer — Key Finding:**

The Property 1 models perform poorly when evaluated against Property 2 labels. The high false negative count confirms that models trained to reject couples and pet owners will incorrectly reject those same applicants even when the new property allows them.

This demonstrates that **trained models are preference-specific** — a single model cannot serve multiple property types with different landlord rules. The system requires either:
- A dedicated model per property configuration, or
- A configuration layer where preferences are parameterised and the model is retrained accordingly

The pipeline itself (feature extraction, TF-IDF/BERT processing, evaluation) is fully reusable. Only the training data labels change.

### 13c — New Property Dataset & Retrain
Generate a new dataset of ~150 applicants for a 2-bed property (max 3 occupants: 2 adults + child, pets allowed, min salary €40k). Retrain all three models and compare with Property 1 results.

In [ ]:
# Download Property 2 dataset from GitHub
p2_url = 'https://github.com/B00766693/College_Rental_Suitability/raw/main/Property2_Dataset.xlsx'
urllib.request.urlretrieve(p2_url, 'Property2_Dataset.xlsx')

df_p2 = pd.read_excel('Property2_Dataset.xlsx')
print(f'Property 2 dataset loaded: {len(df_p2)} rows')
print(f'  Columns: {list(df_p2.columns)}')
print(f'  num_adults distribution: {dict(df_p2["num_adults"].value_counts().sort_index())}')
print(f'  pets_info distribution: {dict(df_p2["pets_info"].value_counts().sort_index())}')
print(f'  decision distribution: {dict(df_p2["decision"].value_counts().sort_index())}')
df_p2.head(5)

In [ ]:
# Apply feature engineering to Property 2 dataset (decision column already present)
df_p2 = extract_derived_features(df_p2)

print(f'Property 2 class distribution:')
print(f'  Suitable: {(df_p2["decision"]==1).sum()} ({(df_p2["decision"]==1).mean()*100:.1f}%)')
print(f'  Not Suitable: {(df_p2["decision"]==0).sum()} ({(df_p2["decision"]==0).mean()*100:.1f}%)')
print()

# Show sample with derived features
df_p2[['num_adults', 'Total_Occupancy_Count', 'pets_info', 'Pet_Binary',
        'Professional_Status', 'Salary', 'decision']].sample(10, random_state=42)

In [ ]:
# Prepare NB/DT features for Property 2
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()

STOP_WORDS_P2 = {
    'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'from', 'with', 'of', 'by', 'about',
    'into', 'near', 'hello', 'hi', 'hey', 'dear', 'regards', 'thanks', 'thank', 'sincerely',
    'please', 'is', 'am', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'can', 'may',
    'also', 'very', 'really', 'currently', 'recently', 'just', 'already', 'well', 'hope',
    'it', 'its', 'this', 'that', 'so', 'too', 'up', 'out', 'if', 'as', 'or', 'but', 'and',
    'he', 'she', 'they', 'them', 'their', 'his', 'her', 'him', 'you', 'your',
    'which', 'what', 'who', 'how', 'when', 'where', 'why', 'removed',
}

def preprocess_text(text):
    if not text or str(text).strip() == '':
        return ''
    msg = str(text)
    msg = re.sub(r'€', '', msg)
    msg = re.sub(r'\d+', '', msg)
    msg = re.sub(r'non[\-\s]smok\w*', 'nonsmoker', msg, flags=re.IGNORECASE)
    msg = re.sub(r'full[\-\s]time', 'fulltime', msg, flags=re.IGNORECASE)
    msg = re.sub(r'long[\-\s]term', 'longterm', msg, flags=re.IGNORECASE)
    msg = msg.lower()
    msg = re.sub(r'[^a-z\s]', ' ', msg)
    tokens = word_tokenize(msg)
    filtered = [t for t in tokens if t not in STOP_WORDS_P2 and len(t) > 1]
    lemmatised = []
    for t in filtered:
        lem_n = lemmatizer.lemmatize(t, pos='n')
        lem_v = lemmatizer.lemmatize(t, pos='v')
        lemmatised.append(lem_v if len(lem_v) < len(lem_n) else lem_n)
    return ' '.join(lemmatised)

df_p2['message_preprocessed'] = df_p2['message_snippet'].apply(preprocess_text)

# TF-IDF
tfidf_p2 = TfidfVectorizer(max_features=500, min_df=3, max_df=0.95)
X_tfidf_p2 = tfidf_p2.fit_transform(df_p2['message_preprocessed'])

# Combine with structured features
X_struct_p2 = df_p2[structured_features].fillna(0).values
X_combined_p2 = hstack([X_tfidf_p2, X_struct_p2])
y_p2 = df_p2['decision']

# 70/30 split
X_train_p2, X_test_p2, y_train_p2, y_test_p2 = train_test_split(
    X_combined_p2, y_p2, test_size=0.3, random_state=42, stratify=y_p2
)

print(f'Property 2 TF-IDF matrix: {X_tfidf_p2.shape}')
print(f'Training: {X_train_p2.shape[0]} | Test: {X_test_p2.shape[0]}')
print(f'Train distribution: {dict(y_train_p2.value_counts())}')
print(f'Test distribution: {dict(y_test_p2.value_counts())}')

In [ ]:
# Train NB and DT on Property 2 data
nb_p2 = MultinomialNB()
nb_p2.fit(X_train_p2, y_train_p2)
nb_pred_p2 = nb_p2.predict(X_test_p2)
nb_proba_p2 = nb_p2.predict_proba(X_test_p2)[:, 1]

dt_p2 = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_leaf=5)
dt_p2.fit(X_train_p2, y_train_p2)
dt_pred_p2 = dt_p2.predict(X_test_p2)
dt_proba_p2 = dt_p2.predict_proba(X_test_p2)[:, 1]

print('Naïve Bayes — Property 2:')
print(classification_report(y_test_p2, nb_pred_p2, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(y_test_p2, nb_proba_p2):.4f}')

print('\nDecision Tree — Property 2:')
print(classification_report(y_test_p2, dt_pred_p2, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(y_test_p2, dt_proba_p2):.4f}')

In [ ]:
# Train BERT on Property 2 data
texts_p2 = df_p2['message_snippet'].tolist()
labels_p2 = df_p2['decision'].tolist()

train_texts_p2, test_texts_p2, train_labels_p2, test_labels_p2 = train_test_split(
    texts_p2, labels_p2, test_size=0.3, random_state=42, stratify=labels_p2
)

train_dataset_p2 = RentalDataset(train_texts_p2, train_labels_p2)
test_dataset_p2 = RentalDataset(test_texts_p2, test_labels_p2)
train_loader_p2 = DataLoader(train_dataset_p2, batch_size=16, shuffle=True)
test_loader_p2 = DataLoader(test_dataset_p2, batch_size=16, shuffle=False)

# Fresh BERT model for Property 2
model_p2 = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model_p2.to(device)

optimizer_p2 = AdamW(model_p2.parameters(), lr=2e-5, eps=1e-8)
total_steps_p2 = len(train_loader_p2) * EPOCHS
scheduler_p2 = get_linear_schedule_with_warmup(optimizer_p2, num_warmup_steps=0,
                                                num_training_steps=total_steps_p2)

# Fine-tune
for epoch in range(EPOCHS):
    model_p2.train()
    total_loss = 0
    for batch in train_loader_p2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer_p2.zero_grad()
        outputs = model_p2(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_p2.parameters(), 1.0)
        optimizer_p2.step()
        scheduler_p2.step()
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader_p2):.4f}')

# Evaluate
model_p2.eval()
bert_preds_p2, bert_proba_p2, bert_true_p2 = [], [], []
with torch.no_grad():
    for batch in test_loader_p2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model_p2(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        bert_preds_p2.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        bert_proba_p2.extend(probs[:, 1].cpu().numpy())
        bert_true_p2.extend(batch['label'].numpy())

print('\nBERT — Property 2:')
print(classification_report(bert_true_p2, bert_preds_p2, target_names=['Not Suitable', 'Suitable']))
print(f'ROC AUC: {roc_auc_score(bert_true_p2, bert_proba_p2):.4f}')

In [ ]:
# Cross-property comparison
print('=' * 80)
print('CROSS-PROPERTY MODEL COMPARISON')
print('=' * 80)

comparison_cross = pd.DataFrame({
    'Model': ['NB (P1)', 'DT (P1)', 'BERT (P1)',
              'NB (P2)', 'DT (P2)', 'BERT (P2)'],
    'Property': ['1-Bed', '1-Bed', '1-Bed',
                 '2-Bed', '2-Bed', '2-Bed'],
    'F1': [
        f1_score(y_test, nb_pred),
        f1_score(y_test, dt_pred),
        f1_score(bert_true, bert_preds),
        f1_score(y_test_p2, nb_pred_p2),
        f1_score(y_test_p2, dt_pred_p2),
        f1_score(bert_true_p2, bert_preds_p2),
    ],
    'Precision': [
        precision_score(y_test, nb_pred),
        precision_score(y_test, dt_pred),
        precision_score(bert_true, bert_preds),
        precision_score(y_test_p2, nb_pred_p2),
        precision_score(y_test_p2, dt_pred_p2),
        precision_score(bert_true_p2, bert_preds_p2),
    ],
    'ROC AUC': [
        roc_auc_score(y_test, nb_proba),
        roc_auc_score(y_test, dt_proba),
        roc_auc_score(bert_true, bert_proba),
        roc_auc_score(y_test_p2, nb_proba_p2),
        roc_auc_score(y_test_p2, dt_proba_p2),
        roc_auc_score(bert_true_p2, bert_proba_p2),
    ]
}).round(4)

print(comparison_cross.to_string(index=False))

# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
width = 0.35

p1_f1 = comparison_cross[comparison_cross['Property']=='1-Bed']['F1'].values
p2_f1 = comparison_cross[comparison_cross['Property']=='2-Bed']['F1'].values

bars1 = ax.bar(x - width/2, p1_f1, width, label='Property 1 (1-Bed)', color='#6366f1')
bars2 = ax.bar(x + width/2, p2_f1, width, label='Property 2 (2-Bed)', color='#f472b6')

ax.set_xlabel('Model')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score Comparison Across Property Types')
ax.set_xticks(x)
ax.set_xticklabels(['Naïve Bayes', 'Decision Tree', 'BERT'])
ax.legend()
ax.set_ylim(0, 1.1)
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.2f}', ha='center', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cross_property_comparison.pdf', format='pdf', bbox_inches='tight')
plt.show()

### Section 13 — Key Findings

1. **The preference engine works:** The `label_by_preferences()` function successfully re-labels the same dataset under different landlord rules, demonstrating that the system can be parameterised for any property type.

2. **Models do not transfer across property types:** The zero-shot transfer test confirms that models trained on Property 1 rules (reject couples, reject pets) perform poorly when evaluated against Property 2 rules (allow couples, allow pets). The high false negative rate shows the models incorrectly reject applicants who are suitable under the new rules.

3. **The pipeline is reusable:** When retrained on Property 2 data with Property 2 labels, the same NB, DT, and BERT pipeline produces a working model. The feature engineering, TF-IDF/BERT processing, and evaluation framework required no changes — only the training data and labels changed.

4. **Practical implication:** A production system would need a configuration layer where the letting agent specifies preferences per property (max occupants, pets policy, minimum salary). The system would then use the first batch of manually-labelled enquiries to train a property-specific model, after which screening is automated. The pipeline code is shared across all properties; only the trained model weights and preference parameters differ.